# HTO Correction Angle — Inter-Specialist Agreement

Computes the **Miniaci correction angle** for every image, every leg (left/right image hemisphere), and each of the three specialists, directly from their annotated landmarks — then quantifies agreement between every pair of specialists.

**Construction (per leg):** ankle centre = midpoint(`ankle_inner`, `ankle_outer`); Fujisawa target = `knee_inner` + 0.625·(`knee_outer` − `knee_inner`); the desired mechanical axis (`femur_head` → Fujisawa) is intersected with the ankle level; the correction angle is the angle at the osteotomy hinge (`ost_point`) between the rays to the current and desired ankle positions.

Angles are computed in original image pixels (the angle is invariant to the uniform scale/shift, so no rescaling is needed). Any missing/zero landmark ⇒ the angle is `NaN` for that case. *Hemisphere* means **image** left/right (assigned by landmark x); the patient's anatomical side is mirrored.

**Outputs:** a per-image table of angles; per-pair **mean / std / max** angle error (plus signed bias, 95% limits of agreement); and the 3-rater ICC.


In [1]:
import json, collections, itertools
import numpy as np
import pandas as pd

ANNOTATION_PATH = "/tf/data/hto/xrays/hto_annotations.json"     # adjust if your file lives elsewhere
FUJISAWA_RATIO  = 0.625                        # weight-bearing target along the plateau (medial -> lateral)

coco          = json.load(open(ANNOTATION_PATH))
images        = {im["id"]: im for im in coco["images"]}
cat_kps       = {c["id"]: c["keypoints"] for c in coco["categories"]}
fname_to_meta = {im["file_name"]: im.get("correction_angle") for im in coco["images"]}
SPECIALIST_FIELDS = ["keypoints_specialist1", "keypoints_specialist2", "keypoints_specialist3"]
SPECIALIST_LABELS = ["S1", "S2", "S3"]

anns_by_img = collections.defaultdict(list)
for a in coco["annotations"]:
    anns_by_img[a["image_id"]].append(a)

print(f"Loaded {len(images)} images, {len(coco['annotations'])} annotations, "
      f"{len(SPECIALIST_LABELS)} specialists.")
print("Categories:", {c['id']: c['keypoints'] for c in coco['categories']})

Loaded 70 images, 413 annotations, 3 specialists.
Categories: {1: ['femur_head'], 2: ['knee_inner', 'ost_point', 'knee_outer'], 3: ['ankle_inner', 'ankle_outer']}


In [2]:
SIDE_KEYS = ["femur_head", "knee_inner", "ost_point", "knee_outer", "ankle_inner", "ankle_outer"]

def _triplets(ann, field):
    """{keypoint_name: (x, y, v)} for one annotation under one specialist field."""
    kp = ann[field]; names = cat_kps[ann["category_id"]]
    return {names[i]: (kp[3*i], kp[3*i+1], kp[3*i+2]) for i in range(len(names))}

def _representative_x(ann):
    """Specialist-independent x for assigning an annotation to the left/right image hemisphere."""
    xs = [kp[i] for f in SPECIALIST_FIELDS for kp in [ann[f]]
          for i in range(0, len(kp), 3) if kp[i] != 0 or kp[i+1] != 0]
    return float(np.mean(xs)) if xs else ann["bbox"][0] + ann["bbox"][2] / 2.0

def miniaci_correction_angle(points, ratio=FUJISAWA_RATIO):
    """Miniaci correction angle (degrees) from one leg's 6 landmarks.
    points: {name: (x, y)}; returns np.nan if any landmark is missing or (0,0)."""
    pts = {}
    for n in SIDE_KEYS:
        if n not in points:
            return np.nan
        x, y = points[n]
        if x == 0 and y == 0:
            return np.nan
        pts[n] = np.array([x, y], float)
    fh, ki, op, ko, ai, ao = (pts[n] for n in SIDE_KEYS)
    ankle_c  = (ai + ao) / 2.0
    fujisawa = ki + ratio * (ko - ki)
    dy, dx   = fujisawa[1] - fh[1], fujisawa[0] - fh[0]
    if abs(dy) < 1e-9:
        return np.nan
    target_x = fh[0] + (ankle_c[1] - fh[1]) * dx / dy
    target   = np.array([target_x, ankle_c[1]])
    v_orig, v_target = ankle_c - op, target - op
    cross = v_orig[0]*v_target[1] - v_orig[1]*v_target[0]
    dot   = v_orig[0]*v_target[0] + v_orig[1]*v_target[1]
    return float(np.degrees(abs(np.arctan2(cross, dot))))

In [3]:
# ---- compute the correction angle for every (image, hemisphere, specialist) ----
records, angles = [], {}
for img_id, anns in anns_by_img.items():
    im = images[img_id]; fname = im["file_name"]; mid_x = im["width"] / 2.0
    by_cat = collections.defaultdict(list)
    for a in anns:
        by_cat[a["category_id"]].append(a)

    hemi_anns = {"Left": {}, "Right": {}}
    for cid, lst in by_cat.items():
        ordered = sorted(lst, key=_representative_x)
        if len(ordered) == 2:                         # usual case: one leg on each side
            hemi_anns["Left"][cid], hemi_anns["Right"][cid] = ordered[0], ordered[1]
        else:                                         # 1 (or >2) of this category: split by midpoint
            for a in ordered:
                hemi_anns["Left" if _representative_x(a) < mid_x else "Right"][cid] = a

    for hemi in ("Left", "Right"):
        row = {"file_name": fname, "hemisphere": hemi}
        for field, label in zip(SPECIALIST_FIELDS, SPECIALIST_LABELS):
            pts = {}
            for cid in (1, 2, 3):
                if cid in hemi_anns[hemi]:
                    for n, (x, y, v) in _triplets(hemi_anns[hemi][cid], field).items():
                        pts[n] = (x, y)
            row[label] = miniaci_correction_angle(pts)
        angles[(fname, hemi)] = {l: row[l] for l in SPECIALIST_LABELS}
        records.append(row)

long_df = pd.DataFrame(records)[["file_name", "hemisphere"] + SPECIALIST_LABELS]

# ---- wide table: one row per image, Left/Right x each specialist ----
wide = long_df.pivot(index="file_name", columns="hemisphere", values=SPECIALIST_LABELS)
wide.columns = [f"{hemi[0]}_{sp}" for sp, hemi in wide.columns]      # L_S1, R_S1, ...
wide = wide.reindex(sorted(wide.columns), axis=1).reset_index()
wide["img_corr_angle(meta)"] = wide["file_name"].map(fname_to_meta)  # file metadata, NOT the Miniaci value

pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda v: f"{v:.2f}")
n_full = long_df[SPECIALIST_LABELS].notna().all(axis=1).sum()
print(f"Per-image Miniaci correction angles (degrees) — {len(wide)} images, "
      f"{n_full}/{len(long_df)} hemispheres fully annotated by all specialists.\n"
      f"Columns: L_/R_ = left/right image hemisphere; S1/S2/S3 = specialist.\n")
display(wide)

try:
    wide.to_csv("hto_angles_per_image.csv", index=False)
    long_df.to_csv("hto_angles_long.csv", index=False)
    print("\nSaved: hto_angles_per_image.csv, hto_angles_long.csv")
except OSError as e:
    print(f"\n(CSV not written here: {e}. The table is available as `wide` / `long_df`.)")

Per-image Miniaci correction angles (degrees) — 70 images, 107/140 hemispheres fully annotated by all specialists.
Columns: L_/R_ = left/right image hemisphere; S1/S2/S3 = specialist.



,file_name,L_S1,L_S2,L_S3,R_S1,R_S2,R_S3,img_corr_angle(meta)
0,10_0.png,NaN,NaN,NaN,NaN,NaN,NaN,2.10
1,11_0.png,7.00,6.89,6.36,8.10,7.95,8.38,0.56
2,12_0.png,NaN,NaN,NaN,NaN,NaN,NaN,1.19
3,13_0.png,1.62,1.38,1.58,2.72,3.13,3.39,1.06
4,14_0.png,0.41,0.37,1.15,1.03,1.23,1.28,2.34
5,15_0.png,4.65,5.19,5.05,5.23,5.65,6.03,2.19
6,16_0.png,NaN,NaN,NaN,NaN,NaN,NaN,2.73
7,17_0.png,4.34,4.19,3.89,4.82,4.91,5.13,0.72
8,18_0.png,8.80,8.69,8.35,4.88,5.24,5.55,1.55
9,19_0.png,15.22,15.09,14.91,15.66,16.05,16.27,0.57



Saved: hto_angles_per_image.csv, hto_angles_long.csv


In [4]:
# ---- pairwise inter-specialist agreement on the correction angle ----
def pair_stats(angles, hemi_filter=None):
    rows = []
    for a, b in itertools.combinations(SPECIALIST_LABELS, 2):
        diffs = np.array([d[a] - d[b] for (f, h), d in angles.items()
                          if (hemi_filter in (None, h)) and not (np.isnan(d[a]) or np.isnan(d[b]))])
        ad = np.abs(diffs)
        sd = diffs.std(ddof=1) if len(diffs) > 1 else np.nan
        rows.append({
            "pair":          f"{a}-{b}",
            "n":             len(diffs),
            "mean_abs_err":  ad.mean(),
            "std_abs_err":   ad.std(ddof=1) if len(ad) > 1 else np.nan,
            "max_abs_err":   ad.max(),
            "bias(signed)":  diffs.mean(),
            "LoA_lower":     diffs.mean() - 1.96 * sd,
            "LoA_upper":     diffs.mean() + 1.96 * sd,
        })
    return pd.DataFrame(rows)

print("=== Pairwise inter-specialist correction-angle error (degrees) — ALL hemispheres ===")
overall = pair_stats(angles)
display(overall.round(3))

print("\n=== Left image hemisphere ===")
display(pair_stats(angles, "Left").round(3))
print("=== Right image hemisphere ===")
display(pair_stats(angles, "Right").round(3))

try:
    overall.to_csv("hto_pairwise_specialist_error.csv", index=False)
    print("Saved: hto_pairwise_specialist_error.csv")
except OSError as e:
    print(f"(CSV not written here: {e}. Table available as `overall`.)")
print("\nmean_abs_err / std_abs_err / max_abs_err answer the request directly; "
      "bias and LoA (Bland-Altman 95% limits of agreement) are added for the inter-observer write-up.")

=== Pairwise inter-specialist correction-angle error (degrees) — ALL hemispheres ===


,pair,n,mean_abs_err,std_abs_err,max_abs_err,bias(signed),LoA_lower,LoA_upper
0,S1-S2,107,0.32,0.24,1.29,0.01,-0.78,0.79
1,S1-S3,107,0.33,0.23,1.16,-0.01,-0.80,0.79
2,S2-S3,107,0.30,0.20,0.85,-0.01,-0.73,0.71



=== Left image hemisphere ===


,pair,n,mean_abs_err,std_abs_err,max_abs_err,bias(signed),LoA_lower,LoA_upper
0,S1-S2,54,0.30,0.22,0.99,0.11,-0.60,0.82
1,S1-S3,54,0.33,0.20,0.82,0.24,-0.37,0.84
2,S2-S3,54,0.30,0.21,0.79,0.13,-0.56,0.81


=== Right image hemisphere ===


,pair,n,mean_abs_err,std_abs_err,max_abs_err,bias(signed),LoA_lower,LoA_upper
0,S1-S2,53,0.34,0.26,1.29,-0.10,-0.92,0.71
1,S1-S3,53,0.33,0.26,1.16,-0.25,-0.91,0.40
2,S2-S3,53,0.31,0.20,0.85,-0.15,-0.81,0.51


Saved: hto_pairwise_specialist_error.csv

mean_abs_err / std_abs_err / max_abs_err answer the request directly; bias and LoA (Bland-Altman 95% limits of agreement) are added for the inter-observer write-up.


In [5]:
# ---- 3-rater intraclass correlation (single measure), on complete cases ----
def icc(matrix):
    """matrix: [n_subjects, k_raters], no NaN. Returns ICC(2,1) and ICC(3,1) (single rater)."""
    X = np.asarray(matrix, float); n, k = X.shape
    grand = X.mean()
    SSR = k * ((X.mean(axis=1) - grand) ** 2).sum()      # between subjects
    SSC = n * ((X.mean(axis=0) - grand) ** 2).sum()      # between raters
    SST = ((X - grand) ** 2).sum()
    SSE = SST - SSR - SSC
    MSR = SSR / (n - 1); MSC = SSC / (k - 1); MSE = SSE / ((n - 1) * (k - 1))
    icc21 = (MSR - MSE) / (MSR + (k - 1) * MSE + (k / n) * (MSC - MSE))   # two-way random, absolute agreement
    icc31 = (MSR - MSE) / (MSR + (k - 1) * MSE)                          # two-way mixed, consistency
    return icc21, icc31

complete = np.array([[d[l] for l in SPECIALIST_LABELS] for d in angles.values()
                     if not any(np.isnan(d[l]) for l in SPECIALIST_LABELS)])
icc21, icc31 = icc(complete)
print(f"Complete cases (all 3 specialists present): n = {len(complete)}")
print(f"ICC(2,1)  two-way random, absolute agreement, single rater : {icc21:.4f}")
print(f"ICC(3,1)  two-way mixed,  consistency,        single rater : {icc31:.4f}")
print(f"\nMean correction angle across specialists: {complete.mean():.2f}\u00b0 "
      f"(range {complete.min():.2f}\u2013{complete.max():.2f}\u00b0)")
print("\nThis ICC (and the pairwise table above) is the inter-observer variability that "
      "anchors the 'within inter-observer agreement' claim for an automated model.")

Complete cases (all 3 specialists present): n = 107
ICC(2,1)  two-way random, absolute agreement, single rater : 0.9963
ICC(3,1)  two-way mixed,  consistency,        single rater : 0.9962

Mean correction angle across specialists: 6.94° (range 0.00–24.99°)

This ICC (and the pairwise table above) is the inter-observer variability that anchors the 'within inter-observer agreement' claim for an automated model.
